DATASET CONFIGURATION

In [1]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch
from torch import nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os
import torch


from cesnet_datazoo.datasets import CESNET_TLS22
from cesnet_datazoo.config import DatasetConfig, AppSelection
from cesnet_datazoo.config import ScalerEnum

import random
import numpy as np

dataset = CESNET_TLS22("../datasets/CESNET-TLS22/", size="XS")
dataset_config = DatasetConfig(
    dataset=dataset,
    apps_selection=AppSelection.TOPX_KNOWN,
    apps_selection_topx=10,
    train_period_name="W-2021-40",
    test_period_name="W-2021-41",
    psizes_scaler=ScalerEnum.MINMAX,
    ipt_scaler=ScalerEnum.MINMAX,
    flowstats_scaler= ScalerEnum.MINMAX,
    train_dataloader_order= 'random',
    
    
    batch_size=32,
)

dataset.set_dataset_config_and_initialize(dataset_config)

train_dl = dataset.get_train_dataloader()
val_dl = dataset.get_val_dataloader()
test_dl = dataset.get_test_dataloader()



PRETEXT TASKS

In [14]:
def mask_data(sequences, mask_percentage=0.15):
    masked_sequences = sequences.clone()
    num_to_mask = int(torch.ceil(torch.tensor(mask_percentage) * sequences.shape[1] * sequences.shape[2]))
    
    mask_indices = [(i, torch.randint(0, sequences.shape[1], (num_to_mask,)), 
                     torch.randint(0, sequences.shape[2], (num_to_mask,))) 
                    for i in range(sequences.shape[0])]
    
    for i, times, features in mask_indices:
        masked_sequences[i, times, features] = 0  # Zero masking
    device = sequences.device
    return masked_sequences.to(device)


In [13]:
def shuffle_segments(sequences, num_segments=6):
    """
    Shuffles segments within each sequence while keeping the order of elements within each segment intact.

    Parameters:
    - sequences: Tensor of shape [batch_size, seq_len], representing sequences of data points.
    - num_segments: Number of segments to divide each sequence into.

    Returns:
    - shuffled_sequences: Tensor with the same shape as input, with segments shuffled within each sequence.
    - shuffled_order: Tensor containing the shuffled order of segments within each sequence.
    - original_order: Tensor containing the original order of segments within each sequence.
    """
    batch_size, seq_len = sequences.shape
    segment_length = seq_len // num_segments
    
    # Ensure that the sequence can be evenly divided into segments
    assert seq_len % num_segments == 0, "seq_len must be divisible by num_segments"

    device = sequences.device  # Get the device of the input sequences

    # Reshape sequences into segments
    reshaped_sequences = sequences.view(batch_size, num_segments, segment_length)
    
    # Generate a unique shuffled order for segments within each sequence
    shuffled_order = torch.stack([torch.randperm(num_segments, device=device) for _ in range(batch_size)])
    
    # Shuffle segments within each sequence using the generated order
    shuffled_sequences = torch.gather(reshaped_sequences, 1, shuffled_order.unsqueeze(-1).expand(-1, -1, segment_length))
    
    # Return to original shape
    shuffled_sequences = shuffled_sequences.view(batch_size, seq_len)
    
    # Generate original order tensor for reference
    original_order = torch.arange(num_segments, device=device).repeat(batch_size, 1)
    
    return shuffled_sequences.to(device), shuffled_order.to(device), original_order.to(device)


SSL MODEL

In [12]:
class CombinedPretextModel(nn.Module):
    def __init__(self, num_layers, hidden_dim_order,hidden_dim_mask, seq_len, num_segments):
        super(CombinedPretextModel, self).__init__()
        self.num_segments = num_segments
        self.hidden_dim_order = hidden_dim_order
        self.seq_len = seq_len
        self.num_layers = num_layers
        self.hidden_dim_mask = hidden_dim_mask

        #LSTM mask
        self.lstm_mask = nn.LSTM(3, hidden_dim_mask, num_layers, batch_first=True,bidirectional=True)
        #fully connected for mask
        self.fc_mask = nn.Linear(hidden_dim_mask*2, 3)  # Output size matches input feature size
    
         # Pooling layer for mask features
        self.pooling_mask = nn.AvgPool1d(kernel_size=4, stride=4)
       
        # LSTM for sequence order correction
        self.lstm_order = nn.LSTM(1, hidden_dim_order, batch_first=True, bidirectional=True)
        
        # Fully connected layer for sequence order correction task
        self.fc_order = nn.Linear(hidden_dim_order * 2, num_segments * num_segments)
        
        #self.num_feature_embeddings = nn.Linear(in)


    def forward(self, x_masked, x_reordered):
        # Process x_reordered for sequence order correction
        x_reordered = x_reordered.unsqueeze(-1)  # Add feature dimension
        lstm_out_order, _ = self.lstm_order(x_reordered)
        last_hidden_order = lstm_out_order[:, -1, :]
        predictions_order = self.fc_order(last_hidden_order)
        predictions_order = predictions_order.view(-1, self.num_segments, self.num_segments)


        h0 = torch.zeros(self.num_layers*2, x_masked.size(0), self.hidden_dim_mask).to(x_masked.device)
        c0 = torch.zeros(self.num_layers*2, x_masked.size(0), self.hidden_dim_mask).to(x_masked.device)
        
        # Forward propagate LSTM
        masking_predictions, _ = self.lstm_mask(x_masked, (h0, c0))  # out: tensor of shape (batch_size, seq_length, hidden_size)
        
        # Decode the hidden state of all time steps
        masking_predictions = self.fc_mask(masking_predictions)
        return masking_predictions, predictions_order 
    def extract_features(self, x_masked, x_reordered):
        x_reordered = x_reordered.unsqueeze(-1)
        lstm_out_order, _ = self.lstm_order(x_reordered)
        last_hidden_order = lstm_out_order[:, -1, :]
        

        h0 = torch.zeros(self.num_layers*2, x_masked.size(0), self.hidden_dim_mask).to(x_masked.device)
        c0 = torch.zeros(self.num_layers*2, x_masked.size(0), self.hidden_dim_mask).to(x_masked.device)
        masking_predictions, _ = self.lstm_mask(x_masked, (h0, c0))
        # Apply pooling to mask features for feature extraction
        masking_predictions_pooled = self.pooling_mask(masking_predictions.permute(0, 2, 1)).permute(0, 2, 1)
        
        # Return the LSTM outputs directly as features
        return masking_predictions_pooled, last_hidden_order

    

CUSTOM LOSS FUNCTION

In [15]:
def sequence_order_loss(predictions, ground_truth):
    """
    Custom loss function for sequence order correction task.
    
    Args:
    - predictions: Tensor of shape [batch_size, num_features, num_segments, num_segments]
    - ground_truth: Tensor representing the ground truth order
    
    Returns:
    - loss: Scalar tensor representing the loss value
    """
    # Compute cross-entropy loss for each sequence independently
    losses = []
    for feature in predictions:
            # Create ground truth tensor based on the provided order
   
            gt_seq = ground_truth

            loss_seq = torch.nn.CrossEntropyLoss()(feature, gt_seq)
            losses.append(loss_seq)
            
    # Compute mean loss across all sequences and features in the batch
    
    
    loss = torch.mean(torch.stack(losses))
    
    return loss


SSL TRAINING LOOP 

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = np.arange(0, 1.1, 0.1)

for order_weight in weights:
    mask_weight = 1 - order_weight  
    model = CombinedPretextModel(hidden_dim_order = 64, hidden_dim_mask=200, seq_len=90, num_segments=90, num_layers=2).to(device)
    model_name = f'model_order={order_weight:.1f}_mask={mask_weight:.1f}.pth'

    model.to(device)

    # Loss function
    criterion_mask = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=5, verbose=True)

    # Training parameters
    num_epochs = 30
    patience = 2
    best_val_loss = float('inf')
    epochs_without_improvement = 0

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        train_losses = []
        for i, (_, ppi, additional_features, y) in enumerate(train_dl):  
            ppi = torch.from_numpy(ppi).float().to(device)
            ppi = ppi.transpose(1, 2)
            original_ppi = ppi.clone()

            optimizer.zero_grad()
            masked_ppi = mask_data(ppi)  # function to mask data

            ppi = ppi.flatten(start_dim=-2)

            shuffled_seqence, shuffled_data, original_orders = shuffle_segments(ppi,90)
            
            shuffled_seqence.to(device)
            
            predicted_masks, predicted_order = model(masked_ppi, shuffled_seqence)

            loss_order = sequence_order_loss(predicted_order, original_orders[0])  
            loss_mask = criterion_mask(predicted_masks, original_ppi)
            
            total_loss = loss_order * order_weight + loss_mask * mask_weight
            
            
            train_losses.append(total_loss.item())
            total_loss.backward()
            optimizer.step()

            if i % 1000 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i}], Loss: {total_loss.item()}")

        # Validation phase
        val_losses = []
        model.eval()
        with torch.no_grad():
            for i, (_, ppi, additional_features, y) in enumerate(val_dl): 
                ppi = torch.from_numpy(ppi).float().to(device)
                ppi = ppi.transpose(1, 2)
                original_ppi = ppi.clone()
                masked_ppi = mask_data(ppi)  # function to mask data

                ppi = ppi.flatten(start_dim=-2)
                
                shuffled_seqence, shuffled_data, original_orders = shuffle_segments(ppi,90)
                
                shuffled_seqence.to(device)
                
                predicted_masks, predicted_order = model(masked_ppi, shuffled_seqence)

                order_loss_val = sequence_order_loss(predicted_order, original_orders[0])  
            
                mask_loss_val = criterion_mask(predicted_masks, original_ppi)
                
                total_val_loss = mask_loss_val * mask_weight + order_loss_val * order_weight
            

                
                val_losses.append(total_val_loss.item())

        avg_val_loss = np.mean(val_losses)
        print(f"Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_val_loss}")
        scheduler.step(avg_val_loss)

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            # Save the model checkpoint
            torch.save(model.state_dict(), model_name)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print("Early stopping triggered.")
                break
    del model, optimizer, scheduler
    torch.cuda.empty_cache()


/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch [1/30], Step [0], Loss: 0.28574231266975403
Epoch [1/30], Step [1000], Loss: 0.015280704945325851
Epoch [1/30], Step [2000], Loss: 0.012889002449810505
Epoch [1/30], Step [3000], Loss: 0.007977282628417015
Epoch [1/30], Step [4000], Loss: 0.00796778779476881
Epoch [1/30], Step [5000], Loss: 0.005449412856251001
Epoch [1/30], Step [6000], Loss: 0.00457732891663909
Epoch [1/30], Step [7000], Loss: 0.0066586765460669994
Epoch [1/30], Step [8000], Loss: 0.008981814607977867
Epoch [1/30], Step [9000], Loss: 0.006923466455191374
Epoch [1/30], Step [10000], Loss: 0.007546431850641966
Epoch [1/30], Step [11000], Loss: 0.003892390988767147
Epoch [1/30], Step [12000], Loss: 0.007428450044244528
Epoch [1/30], Step [13000], Loss: 0.0024291242007166147
Epoch [1/30], Step [14000], Loss: 0.004042061977088451
Epoch [1/30], Step [15000], Loss: 0.0038815115112811327
Epoch [1/30], Step [16000], Loss: 0.0057325707748532295
Epoch [1/30], Step [17000], Loss: 0.00393840204924345
Epoch [1/30], Step [180

FEATURE EXTRACTION AND DATASET CREATION

In [16]:
from torch.utils.data import Dataset
import os
import torch

class NetFlowDataset(Dataset):
    def __init__(self, features_dir, labels_dir):
        self.features_files = [os.path.join(features_dir, f) for f in sorted(os.listdir(features_dir))]
        self.labels_files = [os.path.join(labels_dir, f) for f in sorted(os.listdir(labels_dir))]
        
        assert len(self.features_files) == len(self.labels_files), "Mismatched number of features and labels files."

    def __len__(self):
        return len(self.features_files)

    def __getitem__(self, idx):
        features = torch.load(self.features_files[idx])
        labels = torch.load(self.labels_files[idx])
        return features, labels


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for order_weight in weights:
    mask_weight = 1 - order_weight      
    model = CombinedPretextModel(hidden_dim_order = 64, hidden_dim_mask=200, seq_len=90, num_segments=90, num_layers=2).to(device)
    model_name = f'model_order={order_weight:.1f}_mask={mask_weight:.1f}.pth'

    
    model.load_state_dict(torch.load(model_name))
    model.eval()

    # Directory setup
    features_dir = f'extracted_features_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}'
    labels_dir = f'extracted_labels_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}'
    os.makedirs(features_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    with torch.no_grad():
        for i, (_,ppi, additional_features, labels) in enumerate(train_dl):
            # Data pre-processing steps you mentioned
            ppi = torch.from_numpy(ppi).transpose(1, 2).to(device)
            labels = torch.from_numpy(labels).to(device)
            flattened_ppi = ppi.flatten(start_dim=-2).to(device)      

            masked_ppi = mask_data(ppi).to(device)
           
            shuffled_sequence, shuffled_data, original_orders = shuffle_segments(flattened_ppi,90)

            # Extract features using the modelhow to 
            mask_features, order_features = model.extract_features(masked_ppi, shuffled_sequence)
            
            mask_features = mask_features.flatten(start_dim=-2)

            concatenated_features = torch.cat((flattened_ppi, order_features, mask_features), dim=1)
            
            # Save the features and labels to dir
            features_path = os.path.join(features_dir, f'features_batch_{i}.pt')
            labels_path = os.path.join(labels_dir, f'labels_batch_{i}.pt')
            torch.save(concatenated_features, features_path)
            torch.save(labels, labels_path)

            if i % 1000 == 0:
                print(f"Batch {i}, features saved to {features_path}, labels saved to {labels_path}")





  
    model = CombinedPretextModel(hidden_dim_order = 64, hidden_dim_mask=200, seq_len=90, num_segments=90, num_layers=2).to(device)
    # Assuming the model and data_loader are already defined and initialized 
    model.load_state_dict(torch.load(model_name))
    model.eval()
    # Directory setup
    features_dir = f'extracted_features_val_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}'
    labels_dir = f'extracted_labels_val_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}'
    os.makedirs(features_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    with torch.no_grad():
        for i, (_,ppi, additional_features, labels) in enumerate(val_dl):
            # Data pre-processing steps you mentioned
            ppi = torch.from_numpy(ppi).transpose(1, 2).to(device)
            labels = torch.from_numpy(labels)
            flattened_ppi = ppi.flatten(start_dim=-2)        
            #masking ppi and negative sequences
            masked_ppi = mask_data(ppi)
            
            
            #shuffling sequences
            shuffled_sequence, shuffled_data, original_orders = shuffle_segments(flattened_ppi,90)
            # Extract features using the model
            mask_features, order_features = model.extract_features(masked_ppi, shuffled_sequence)
            mask_features = mask_features.flatten(start_dim=-2)
            concatenated_features = torch.cat((flattened_ppi, order_features, mask_features), dim=1)
            
            # Save the features and labels to disk
            features_path = os.path.join(features_dir, f'features_batch_val_{i}.pt')
            labels_path = os.path.join(labels_dir, f'labels_batch_val_{i}.pt')
            torch.save(concatenated_features, features_path)
            torch.save(labels, labels_path)

            if i % 1000 == 0:
                print(f"Batch {i}, features saved to {features_path}, labels saved to {labels_path}")


    


    
    model = CombinedPretextModel(hidden_dim_order = 64, hidden_dim_mask=200, seq_len=90, num_segments=90, num_layers=2).to(device)
   
    # Assuming the model and data_loader are already defined and initialized 
    model.load_state_dict(torch.load(model_name))
    model.eval()
    # Directory setup
    features_dir = f'extracted_features_test_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}'
    labels_dir = f'extracted_labels_test_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}'
    os.makedirs(features_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    with torch.no_grad():
        for i, (_,ppi, additional_features, labels) in enumerate(test_dl):
            # Data pre-processing steps you mentioned
            ppi = torch.from_numpy(ppi).transpose(1, 2).to(device)
            labels = torch.from_numpy(labels)
            flattened_ppi = ppi.flatten(start_dim=-2)        
            #masking ppi and negative sequences
            masked_ppi = mask_data(ppi)
            
            
            #shuffling sequences
            shuffled_sequence, shuffled_data, original_orders = shuffle_segments(flattened_ppi,90)
            # Extract features using the model
            mask_features, order_features = model.extract_features(masked_ppi, shuffled_sequence)
            mask_features = mask_features.flatten(start_dim=-2)
            concatenated_features = torch.cat((flattened_ppi, order_features, mask_features), dim=1)
            
            # Save the features and labels to disk
            features_path = os.path.join(features_dir, f'features_batch_test_{i}.pt')
            labels_path = os.path.join(labels_dir, f'labels_batch_test_{i}.pt')
            torch.save(concatenated_features, features_path)
            torch.save(labels, labels_path)

            if i % 1000 == 0:
                print(f"Batch {i}, features saved to {features_path}, labels saved to {labels_path}")

   


Batch 0, features saved to extracted_features_combined_model_order=0.0_mask=1.0/features_batch_0.pt, labels saved to extracted_labels_combined_model_order=0.0_mask=1.0/labels_batch_0.pt
Batch 1000, features saved to extracted_features_combined_model_order=0.0_mask=1.0/features_batch_1000.pt, labels saved to extracted_labels_combined_model_order=0.0_mask=1.0/labels_batch_1000.pt
Batch 2000, features saved to extracted_features_combined_model_order=0.0_mask=1.0/features_batch_2000.pt, labels saved to extracted_labels_combined_model_order=0.0_mask=1.0/labels_batch_2000.pt
Batch 3000, features saved to extracted_features_combined_model_order=0.0_mask=1.0/features_batch_3000.pt, labels saved to extracted_labels_combined_model_order=0.0_mask=1.0/labels_batch_3000.pt
Batch 4000, features saved to extracted_features_combined_model_order=0.0_mask=1.0/features_batch_4000.pt, labels saved to extracted_labels_combined_model_order=0.0_mask=1.0/labels_batch_4000.pt
Batch 5000, features saved to extr

MODEL FOR CLASSIFICATION

In [18]:
class Classifier(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()

        # Encoder: Transforms the input feature vector into a lower-dimensional space
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            

            
           
        )

        # Classifier: Outputs a logit for each class
        self.classifier = nn.Sequential(
            nn.Linear(512, output_size),
            
        )

    def forward(self, x):
        encoded = self.encoder(x)
        logits = self.classifier(encoded)
        return logits
    def extract_embeddings(self,x):
        encoded = self.encoder(x)
        return encoded


CUSTOM LOSS FUNCTION

In [19]:
def class_loss(predictions, ground_truth):
    """
    Custom loss function that calculates loss for each sample individually and then computes the mean.
    
    Args:
    - predictions: Tensor of shape [batch_size, n_classes], with raw scores for each class.
    - ground_truth: Tensor of shape [batch_size], with the indices of the correct class for each sample.
    
    Returns:
    - loss: Scalar tensor representing the mean loss value across the batch.
    """
    # Initialize a tensor to store individual losses
    individual_losses = torch.zeros(predictions.size(0))
    
    for i in range(predictions.size(0)):
        # Compute the loss for each sample individually
        individual_loss = F.cross_entropy(predictions[i].unsqueeze(0), ground_truth[i].unsqueeze(0))
        individual_losses[i] = individual_loss
    
    # Compute the mean of the individual losses
    mean_loss = torch.mean(individual_losses)
    
    return mean_loss


DOWNSTREAM TASK

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for order_weight in weights:
    mask_weight = 1 - order_weight
    dataset_features = NetFlowDataset(f'extracted_features_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}', f'extracted_labels_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}')

    # Initialize DataLoader
    dataloader_features = DataLoader(dataset_features, batch_size=1, shuffle=True)
        
    
    dataset_features_val = NetFlowDataset(f'extracted_features_val_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}', f'extracted_labels_val_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}')

    dataloader_features_val = DataLoader(dataset_features_val, batch_size=1, shuffle=True)
        
    dataset_features_test = NetFlowDataset(f'extracted_features_test_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}', f'extracted_labels_test_combined_model_order={order_weight:.1f}_mask={mask_weight:.1f}')


    dataloader_features_test = DataLoader(dataset_features_test, batch_size=1, shuffle=True)



    model_name = f'model_order={order_weight:.1f}_mask={mask_weight:.1f}.pth'
    classifier = Classifier(input_size=3018, output_size=10).to(device)
    criterion = nn.CrossEntropyLoss()  # Appropriate for classification tasks
    optimizer = optim.Adam(classifier.parameters(), lr=0.001)

    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=2, verbose=True)

    num_epochs = 30
    patience = 2
    best_val_loss = float('inf')
    epochs_without_improvement = 0
 
#################################TRAIN############################################
    for epoch in range(num_epochs):
        classifier.train()
        train_losses = []

        for i, (features, labels) in enumerate(dataloader_features):
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            logits = classifier(features)
            logits = logits.squeeze(0)
            labels = labels.flatten(start_dim= -2)
            loss = class_loss(logits, labels)

            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())

            if i % 1000 == 0:
                print(f"Epoch {epoch} Batch {i}, Loss: {loss.item()}")

#################################EVAL############################################
        classifier.eval()
        all_preds = []
        all_labels = []
        total_accuracy = 0
        batches = 0
        val_losses = []
        with torch.no_grad():
            for i, (features, labels) in enumerate(dataloader_features_val):
                features, labels = features.to(device), labels.to(device)
                logits = classifier(features)
                logits = logits.squeeze(0)
                labels = labels.flatten(start_dim= -2)
                
                val_loss = class_loss(logits, labels)
                val_losses.append(val_loss.item())
                predicted_labels = torch.argmax(logits, dim=1)
                apps = dataset.get_known_apps()
                batches +=1
    
            
            
                predicted_apps = [apps[idx] for idx in predicted_labels]
                actual_apps = [apps[idx] for idx in labels]

                
                all_preds.extend(predicted_labels.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
                            # Compare predicted labels with actual labels
                correct_predictions = (predicted_labels == labels).sum().item()
                total_predictions = labels.size(0)
                accuracy = correct_predictions / total_predictions
                total_accuracy += accuracy

                

        avg_val_loss = np.mean(val_losses)
        print(f"Epoch [{epoch+1}/{num_epochs}], Average Validation Loss: {avg_val_loss}")
        print(f"Validation - Batch {i}, Loss: {val_loss.item()}")
        #print(f"Predicted labels: {predicted_apps}")
        #print(f"Actual labels: {actual_apps}\n")
        print(f"Average accuracy: {total_accuracy/batches * 100:.2f}%")
        print(model_name)

        scheduler.step(avg_val_loss)

        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            torch.save(classifier.state_dict(), 'best_classifier.pth')
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Stopping early at epoch {epoch+1}. No improvement in validation loss for {patience} consecutive epochs. Best Loss: {best_val_loss:.4f}.")
                break

    # Evaluate model on validation set again or perform further analysis as needed
    # Load the best model
    print(f"Average accuracy: {total_accuracy/batches * 100:.2f}%")
   


##################################TEST#######################################################

    from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    # Load the best saved model weights
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    classifier = Classifier(input_size=3018, output_size=10).to(device)
    classifier.load_state_dict(torch.load('best_classifier.pth'))
    classifier.eval()  # Set the model to evaluation mode

    test_losses = []
    all_preds_test = []
    all_labels_test = []
    total_accuracy_test = 0
    batches = 0

    with torch.no_grad():
        for i, (features, labels) in enumerate(dataloader_features_test):
            features, labels = features.to(device), labels.to(device)
            logits = classifier(features)
            logits = logits.squeeze(0)
            labels = labels.flatten(start_dim= -2)
            test_loss = class_loss(logits, labels)
            
            
            test_losses.append(test_loss.item())

            predicted_labels = torch.argmax(logits, dim=1)
            all_preds_test.extend(predicted_labels.cpu().numpy())
            all_labels_test.extend(labels.cpu().numpy())

            # Accuracy calculation
            correct_predictions = (predicted_labels == labels).sum().item()
            total_predictions = labels.size(0)
            accuracy = correct_predictions / total_predictions
            total_accuracy_test += accuracy
            batches += 1
            #print(i)

    avg_test_loss = np.mean(test_losses)
    print(f"Average Test Loss: {avg_test_loss}")
    print(f"Average Test Accuracy: {total_accuracy_test/batches * 100:.2f}%")



    # Precision, Recall, and F1 Score
    precision = precision_score(all_labels_test, all_preds_test, average='weighted')
    recall = recall_score(all_labels_test, all_preds_test, average='weighted')
    f1 = f1_score(all_labels_test, all_preds_test, average='weighted')

    print(f'Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')

    # Confusion Matrix
    """  cm = confusion_matrix(all_labels_test, all_preds_test)
    plt.figure(figsize=(30,30))
    sns.heatmap(cm, annot=True, fmt="d", cmap='Blues')
    plt.ylabel('Actual Labels')
    plt.xlabel('Predicted Labels')
    plt.title('Confusion Matrix')
    plt.show() """
    del classifier, optimizer, scheduler
    torch.cuda.empty_cache()

model_order=0.0_mask=1.0.pth
Epoch 0 Batch 0, Loss: 2.300069808959961
Epoch 0 Batch 1000, Loss: 0.1031896248459816
Epoch 0 Batch 2000, Loss: 0.1831703633069992
Epoch 0 Batch 3000, Loss: 0.16578781604766846
Epoch 0 Batch 4000, Loss: 0.33074629306793213
Epoch 0 Batch 5000, Loss: 0.11719350516796112
Epoch 0 Batch 6000, Loss: 0.3082681894302368
Epoch 0 Batch 7000, Loss: 0.3390289545059204
Epoch 0 Batch 8000, Loss: 0.17694279551506042
Epoch 0 Batch 9000, Loss: 0.17269830405712128
Epoch 0 Batch 10000, Loss: 0.3472627103328705
Epoch 0 Batch 11000, Loss: 0.048188649117946625
Epoch 0 Batch 12000, Loss: 0.057303912937641144
Epoch 0 Batch 13000, Loss: 0.24963918328285217
Epoch 0 Batch 14000, Loss: 0.22608010470867157
Epoch 0 Batch 15000, Loss: 0.12539313733577728
Epoch 0 Batch 16000, Loss: 0.15621158480644226
Epoch 0 Batch 17000, Loss: 0.4508362412452698
Epoch 0 Batch 18000, Loss: 0.08437775075435638
Epoch 0 Batch 19000, Loss: 0.26327797770500183
Epoch 0 Batch 20000, Loss: 0.1465926468372345
Epoc

/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 0 Batch 1000, Loss: 0.3766399919986725
Epoch 0 Batch 2000, Loss: 0.14249540865421295
Epoch 0 Batch 3000, Loss: 0.2652091979980469
Epoch 0 Batch 4000, Loss: 0.3476193845272064
Epoch 0 Batch 5000, Loss: 0.2634604871273041
Epoch 0 Batch 6000, Loss: 0.27084413170814514
Epoch 0 Batch 7000, Loss: 0.28863659501075745
Epoch 0 Batch 8000, Loss: 0.2813575267791748
Epoch 0 Batch 9000, Loss: 0.3127801716327667
Epoch 0 Batch 10000, Loss: 0.23904015123844147
Epoch 0 Batch 11000, Loss: 0.09113992750644684
Epoch 0 Batch 12000, Loss: 0.23242685198783875
Epoch 0 Batch 13000, Loss: 0.1099410429596901
Epoch 0 Batch 14000, Loss: 0.055568329989910126
Epoch 0 Batch 15000, Loss: 0.09960726648569107
Epoch 0 Batch 16000, Loss: 0.23926784098148346
Epoch 0 Batch 17000, Loss: 0.3020763695240021
Epoch 0 Batch 18000, Loss: 0.20206134021282196
Epoch 0 Batch 19000, Loss: 0.25227975845336914
Epoch 0 Batch 20000, Loss: 0.22783246636390686
Epoch 0 Batch 21000, Loss: 0.06573355942964554
Epoch [1/30], Average Validat

/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 0 Batch 1000, Loss: 0.38484954833984375
Epoch 0 Batch 2000, Loss: 0.44008857011795044
Epoch 0 Batch 3000, Loss: 0.19882047176361084
Epoch 0 Batch 4000, Loss: 0.19483362138271332
Epoch 0 Batch 5000, Loss: 0.33054399490356445
Epoch 0 Batch 6000, Loss: 0.28812986612319946
Epoch 0 Batch 7000, Loss: 0.16778169572353363
Epoch 0 Batch 8000, Loss: 0.1806846559047699
Epoch 0 Batch 9000, Loss: 0.11488409340381622
Epoch 0 Batch 10000, Loss: 0.2548125684261322
Epoch 0 Batch 11000, Loss: 0.05789456516504288
Epoch 0 Batch 12000, Loss: 0.38623756170272827
Epoch 0 Batch 13000, Loss: 0.42278364300727844
Epoch 0 Batch 14000, Loss: 0.2240796983242035
Epoch 0 Batch 15000, Loss: 0.13008232414722443
Epoch 0 Batch 16000, Loss: 0.32951635122299194
Epoch 0 Batch 17000, Loss: 0.17310874164104462
Epoch 0 Batch 18000, Loss: 0.06403709203004837
Epoch 0 Batch 19000, Loss: 0.06359533220529556
Epoch 0 Batch 20000, Loss: 0.24398066103458405
Epoch 0 Batch 21000, Loss: 0.3270818889141083
Epoch [1/30], Average Vali

...

Epoch 0 Batch 1000, Loss: 0.3046957552433014
Epoch 0 Batch 2000, Loss: 0.26569920778274536
Epoch 0 Batch 3000, Loss: 0.2682306170463562
Epoch 0 Batch 4000, Loss: 0.15527451038360596
Epoch 0 Batch 5000, Loss: 0.3270653486251831
Epoch 0 Batch 6000, Loss: 0.30905893445014954
Epoch 0 Batch 7000, Loss: 0.20279909670352936
Epoch 0 Batch 8000, Loss: 0.12329600751399994
Epoch 0 Batch 9000, Loss: 0.40407949686050415
Epoch 0 Batch 10000, Loss: 0.05259422957897186
Epoch 0 Batch 11000, Loss: 0.40209388732910156
Epoch 0 Batch 12000, Loss: 0.13354071974754333
Epoch 0 Batch 13000, Loss: 0.19227319955825806
Epoch 0 Batch 14000, Loss: 0.2965482771396637
Epoch 0 Batch 15000, Loss: 0.17662130296230316
Epoch 0 Batch 16000, Loss: 0.1224808394908905
Epoch 0 Batch 17000, Loss: 0.34289929270744324
Epoch 0 Batch 18000, Loss: 0.21556256711483002
Epoch 0 Batch 19000, Loss: 0.12057656794786453
Epoch 0 Batch 20000, Loss: 0.10054609924554825
Epoch 0 Batch 21000, Loss: 0.23531493544578552
Epoch [1/30], Average Valid

/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 0 Batch 1000, Loss: 0.46621596813201904
Epoch 0 Batch 2000, Loss: 0.3467831313610077
Epoch 0 Batch 3000, Loss: 0.023921526968479156
Epoch 0 Batch 4000, Loss: 0.028234779834747314
Epoch 0 Batch 5000, Loss: 0.49912503361701965
Epoch 0 Batch 6000, Loss: 0.4503534436225891
Epoch 0 Batch 7000, Loss: 0.2077448070049286
Epoch 0 Batch 8000, Loss: 0.1338249146938324
Epoch 0 Batch 9000, Loss: 0.23461705446243286
Epoch 0 Batch 10000, Loss: 0.21302220225334167
Epoch 0 Batch 11000, Loss: 0.05412622541189194
Epoch 0 Batch 12000, Loss: 0.3083230257034302
Epoch 0 Batch 13000, Loss: 0.10788872092962265
Epoch 0 Batch 14000, Loss: 0.1947993040084839
Epoch 0 Batch 15000, Loss: 0.296549916267395
Epoch 0 Batch 16000, Loss: 0.1568370759487152
Epoch 0 Batch 17000, Loss: 0.3067953884601593
Epoch 0 Batch 18000, Loss: 0.21185635030269623
Epoch 0 Batch 19000, Loss: 0.034574732184410095
Epoch 0 Batch 20000, Loss: 0.1805829107761383
Epoch 0 Batch 21000, Loss: 0.33096519112586975
Epoch [1/30], Average Validati

/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 0 Batch 1000, Loss: 1.0048221349716187
Epoch 0 Batch 2000, Loss: 0.6878544092178345
Epoch 0 Batch 3000, Loss: 0.7162681221961975
Epoch 0 Batch 4000, Loss: 0.5815467834472656
Epoch 0 Batch 5000, Loss: 0.5002779364585876
Epoch 0 Batch 6000, Loss: 0.30198174715042114
Epoch 0 Batch 7000, Loss: 0.5624364018440247
Epoch 0 Batch 8000, Loss: 0.45645418763160706
Epoch 0 Batch 9000, Loss: 0.37431585788726807
Epoch 0 Batch 10000, Loss: 0.26020482182502747
Epoch 0 Batch 11000, Loss: 0.16229583323001862
Epoch 0 Batch 12000, Loss: 0.4339507818222046
Epoch 0 Batch 13000, Loss: 0.2265912890434265
Epoch 0 Batch 14000, Loss: 0.34231796860694885
Epoch 0 Batch 15000, Loss: 0.2979331612586975
Epoch 0 Batch 16000, Loss: 0.19283130764961243
Epoch 0 Batch 17000, Loss: 0.35509419441223145
Epoch 0 Batch 18000, Loss: 0.298100084066391
Epoch 0 Batch 19000, Loss: 0.6290059089660645
Epoch 0 Batch 20000, Loss: 0.21416470408439636
Epoch 0 Batch 21000, Loss: 0.5523077249526978
Epoch [1/30], Average Validation Lo